In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
production_country_schema = StructType(fields= [
                         StructField("movieId", IntegerType(), True),
                         StructField("countryId", IntegerType(), True)
])

In [0]:
production_country_df = spark.read \
                     .schema(production_country_schema) \
                     .option("multiline", True) \
                     .json(f"{bronze_folder_path}/{v_file_date}/production_country")

In [0]:
display(production_country_df)

movieId,countryId
62835,161
62837,214
62838,214
63020,214
63287,214
63492,214
63574,214
64328,214
64499,214
64559,161


In [0]:
production_country_df.count()

1436

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
production_country_final_df = add_ingestion_date(production_country_df) \
                            .withColumnsRenamed({"movieId": "movie_Id", "countryId": "country_Id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(production_country_final_df)

movie_Id,country_Id,ingestion_date,environment,file_date
62835,161,2026-09-11T04:41:08.708039Z,production,2024-12-30
62837,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
62838,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
63020,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
63287,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
63492,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
63574,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
64328,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
64499,214,2026-09-11T04:41:08.708039Z,production,2024-12-30
64559,161,2026-09-11T04:41:08.708039Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "production_country", "file_date", v_file_date)

In [0]:
merge_delta_lake_2(production_country_final_df, "movie_silver", "production_country", "movie_Id", "country_Id", "file_date")

In [0]:
# production_country_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.production_country")

In [0]:
display(spark.read.table("movie_silver.production_country"))

movie_Id,country_Id,ingestion_date,environment,file_date
62835,161,2026-09-11T04:41:09.705983Z,production,2024-12-30
62837,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
62838,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
63020,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
63287,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
63492,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
63574,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
64328,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
64499,214,2026-09-11T04:41:09.705983Z,production,2024-12-30
64559,161,2026-09-11T04:41:09.705983Z,production,2024-12-30


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.production_country
GROUP BY file_date;

file_date,count(1)
2024-12-16,3750
2024-12-23,1250
2024-12-30,1436


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.production_country;

col_name,data_type,comment
movie_Id,int,null
country_Id,int,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,
# Delta Statistics Columns,,


In [0]:
dbutils.notebook.exit("Success")